# RF-ADC interleaving calibration on ZCU208

Hardware check and commissioning procedure for `ol.adc_cal` (`mimo_mts/drivers/adc_calibration.py`),
following PG269 *RF-ADC Calibration Mechanism*.

| Block | Corrects | After RF-ADC startup |
|---|---|---|
| OCB1 | sub-ADC offset | tracks in the background, cannot be frozen |
| OCB2 | offset, sampling-switch skew | set by foreground calibration, then frozen |
| GCB | sub-ADC gain | tracks; needs input above -40 dBFS; can be frozen |
| TSCB | sub-ADC time skew | tracks; needs input above -40 dBFS; can be frozen |

Writing coefficients enables the driver's per-block **override**: written values stay in use until
`release_override()`.

The notebook targets `MIMO_ZCU208` (real input, 4 GSPS, no decimation), where capture index == ADC channel.
Steps marked **[check]** answer the open hardware questions from the merge request; please record their outputs.

### Setup
* A CW signal into the ADC under test, from a signal generator or a DAC loopback cable (see step 3).
* Aim for about -10 dBFS at the ADC, well above the -40 dBFS GCB/TSCB threshold.

In [ ]:
%cd /home/xilinx/jupyter_notebooks/pynq_llrf/

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mimo_mts.overlay import MimoMtsOverlay

%load_ext autoreload
%autoreload 2

In [ ]:
CONFIG = "MIMO_ZCU208"
ADC_CH = 0  # ADC channel under test
# Scratch file for this test. The IOCs default to /etc/mimo_mts/adc_cal.json; do not overwrite it here.
CAL_FILE = Path("/tmp/adc_cal_test.json")
N_AVG = 4  # captures averaged per spectrum

## 1. Load overlay

In [ ]:
ol = MimoMtsOverlay(config=CONFIG)
cal = ol.adc_cal
fs_hz = ol.adc_sampling_rate
print(cal)
print(f"Fs = {fs_hz / 1e9} GHz, ADC mixer: {ol.adc_mixer_nco_freq_mhz} MHz, Nyquist zone {ol.adc_mixer_nco_nyquist}")

## 2. [check] State after RF-ADC startup

Expected per PG269: background calibration tracking, i.e. **not** frozen. The original demo notebook showed every
ADC frozen before any freeze call; record what this board reports.

In [ ]:
blk = cal.block(ADC_CH)
print("raw CalFreeze:", blk.CalFreeze)
print("frozen:       ", cal.is_frozen())
print("cal. mode:    ", cal.calibration_mode())

**[check]** PYNQ coefficient calls with a CFFI struct (`GetCalCoefficients`). This should print four blocks of
8 coefficients for each ADC, without raising.

In [ ]:
print(cal.report())

## 3. Trigger and input signal

Internal trigger, as in `mimo_mts.ipynb`; each trigger refills the ADC capture buffers.

In [ ]:
rm = ol.rf_control.register_map
rm.trig_sel = 0  # internal trigger
rm.trig_period = 125e6  # 2 Hz at 250 MHz
rm.trig_delay = 0
rm.trig_divide = 1
rm.dac_enable = 1
rm.pulse_length = 0xFFF
TRIG_WAIT_S = 0.6  # > one trigger period

**Option A: signal generator** into ADC `ADC_CH`: CW, about -10 dBFS, at the frequency you calibrate for.
Skip the next cell.

**Option B: DAC loopback** (cable from a DAC to ADC `ADC_CH`): play a DC baseband through the DAC NCO.
The DAC output may follow `pulse_length` gating; check that the capture shows a continuous tone in step 4.

In [ ]:
F_CW_MHZ = 1000  # loopback tone frequency

dac_i = np.ones(ol.dac_player.size // 2, dtype=np.int16) * 16383  # about -6 dBFS
dac_q = np.zeros_like(dac_i)
ol.write_dac_iq_buf(dac_i, dac_q)
ol.set_dac_mixer(F_CW_MHZ)

## 4. Spectrum helper

Averages `N_AVG` captures. Interleaving mismatch shows up as spurs at k·Fs/M and k·Fs/M ± f_in (M sub-ADCs).
The table lists the strongest components so the same numbers can be compared at each step.

In [ ]:
def capture(ch=ADC_CH):
    time.sleep(TRIG_WAIT_S)
    i, q = ol.capture_adc_iq_buf()
    return i[ch].astype(np.float64) + 1j * q[ch].astype(np.float64)


def spectrum_dbfs(ch=ADC_CH, n_avg=N_AVG):
    psd = 0
    for _ in range(n_avg):
        x = capture(ch)
        win = np.blackman(len(x))
        psd = psd + np.abs(np.fft.fftshift(np.fft.fft(x * win)) / win.sum()) ** 2
    dbfs = 10 * np.log10(psd / n_avg / 32768.0**2 + 1e-30)
    f_mhz = np.fft.fftshift(np.fft.fftfreq(len(x), 1 / fs_hz)) / 1e6 + ol.adc_mixer_nco_freq_mhz
    return f_mhz, dbfs


def top_components(f_mhz, dbfs, n=8, guard=5):
    """n strongest peaks, ignoring +-guard bins around each one already picked."""
    d = dbfs.copy()
    rows = []
    for _ in range(n):
        k = int(np.argmax(d))
        rows.append((f_mhz[k], dbfs[k]))
        d[max(0, k - guard) : k + guard + 1] = -np.inf
    return rows


def measure(label, ch=ADC_CH):
    f, d = spectrum_dbfs(ch)
    comps = top_components(f, d)
    carrier = comps[0][1]
    spur = max(p for _, p in comps[1:])
    print(f"--- {label}: carrier {carrier:.1f} dBFS, worst listed spur {spur:.1f} dBFS ({carrier - spur:.1f} dBc)")
    for fm, p in comps:
        print(f"    {fm:10.3f} MHz  {p:7.1f} dBFS")
    return f, d


results = {}

In [ ]:
f, d = spectrum_dbfs()
plt.figure(figsize=(10, 4))
plt.plot(f, d)
plt.xlabel("Frequency [MHz]")
plt.ylabel("dBFS")
plt.title(f"ADC{ADC_CH} input")
plt.grid(True)

## 5. Train: background calibration tracking on the CW signal

Make sure the ADCs are tracking, then wait for convergence. PG269 gives time constants of 2^20 to 2^22 samples
(about 1 ms at 4 GSPS); a few seconds is plenty.

In [ ]:
print(cal.unfreeze())
time.sleep(5)
results["tracking"] = measure("tracking")

## 6. Freeze and save

**[check]** `freeze()` should return `True` for every channel, and `CalFreeze` should now show
`DisableFreezePin=1, FreezeCalibration=1, CalFrozen=1`.

In [ ]:
print(cal.freeze())
print("raw CalFreeze:", cal.block(ADC_CH).CalFreeze)
cal.save(CAL_FILE)
results["frozen"] = measure("frozen")

In [ ]:
import json

print(json.dumps(cal.load(CAL_FILE)[f"ADC{ADC_CH}"], indent=1)[:1500])

## 7. Demonstrate: overwrite GCB/TSCB with zeros

This only shows the effect of the coefficients. It writes zeros with the override enabled, so the ADC is
deliberately miscalibrated until step 8.

In [ ]:
cal.write({ADC_CH: {"GCB": [0] * 8, "TSCB": [0] * 8}})
print(cal.report(ADC_CH))
results["zeroed"] = measure("GCB/TSCB zeroed")

## 8. Restore and verify

`restore()` writes GCB/TSCB from the file (override on), freezes, and verifies by read-back. An empty list per
channel means verified. OCB1/OCB2 are left to the hardware by default.

In [ ]:
mismatches = cal.restore(CAL_FILE)
print(mismatches)
results["restored"] = measure("restored")

In [ ]:
print("verify, all blocks:", cal.verify(CAL_FILE))

## 9. Release override and resume tracking

`release_override()` returns all blocks to engine-computed values, then `unfreeze()` resumes GCB/TSCB tracking.

In [ ]:
cal.release_override()
print(cal.unfreeze())
time.sleep(5)
results["released"] = measure("released + tracking")

In [ ]:
plt.figure(figsize=(10, 5))
for label, (f, d) in results.items():
    plt.plot(f, d, label=label, lw=0.8)
plt.xlabel("Frequency [MHz]")
plt.ylabel("dBFS")
plt.title(f"ADC{ADC_CH} calibration states")
plt.legend()
plt.grid(True)

## 10. [check] Restore after a fresh overlay load

The production flow (`AlsLlrfLBL208`): load the overlay, restore, freeze. The spectrum should match the
"restored" result from step 8.

In [ ]:
ol.free()
ol = MimoMtsOverlay(config=CONFIG)
cal = ol.adc_cal
print("after reload, frozen:", cal.is_frozen())
rm = ol.rf_control.register_map
rm.trig_sel, rm.trig_period, rm.trig_delay, rm.trig_divide = 0, 125e6, 0, 1
rm.dac_enable, rm.pulse_length = 1, 0xFFF
# Re-run the Option B cell here if using DAC loopback.

In [ ]:
print(cal.restore(CAL_FILE))
results["reload + restore"] = measure("reload + restore")

## 11. [check] Optional: re-run foreground calibration

`restart_adc_tiles()` resets the ADC tiles (foreground calibration runs again), re-synchronizes MTS (DAC tiles are
reset too) and restores the default mixers. **Mute the ADC input first**: PG269 requires no signal energy at k·Fs/M
during foreground calibration.

Record: does it complete, does `check_mts_latency()` pass, and are freeze state and overrides cleared?

In [ ]:
# Mute the input (signal generator RF off, or the next line for DAC loopback) before running.
zeros = np.zeros(ol.dac_player.size // 2, dtype=np.int16)
ol.write_dac_iq_buf(zeros, zeros)

ol.restart_adc_tiles()
ol.check_mts_latency()
print(ol.report_mts_latency())
print("frozen after restart:", cal.is_frozen())
print(cal.report(ADC_CH))

## Results to report
1. Step 2: raw `CalFreeze` and frozen state after load.
2. Step 2: `cal.report()` ran without error.
3. Step 6: `freeze()` result and raw `CalFreeze`.
4. Steps 5-10: carrier and worst-spur levels for each state.
5. Step 10: restore after reload matches step 8.
6. Step 11: foreground re-run result.